# Test 1


In [4]:
# Install LightGBM via pip
!pip install lightgbm --upgrade


   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   --------------------- ------------------ 0.8/1.5 MB 4.2 MB/s eta 0:00:01
   ---------------------------- ----------- 1.0/1.5 MB 4.1 MB/s eta 0:00:01
   ------------------------------------ --- 1.3/1.5 MB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 1.7 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import h5py
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report, cohen_kappa_score)
from sklearn.utils import resample
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import time



# -------------------- Specificity calculation --------------------
def specificity_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn = cm[0, 0]
    fp = cm[0, 1]
    return tn / (tn + fp) if (tn + fp) > 0 else 0

# -------------------- LightGBM classifier --------------------
def lgb_classifier(X_train, y_train, X_test, y_test, use_scaling=True, params=None):
    print("\n--- LightGBM for Sleep vs Awake classification ---")
    
    # Scale features if needed
    if use_scaling:
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        print("Data scaled with StandardScaler.")
    else:
        X_train_scaled, X_test_scaled = X_train, X_test
        print("Data scaling skipped.")
    
    # Initialize LightGBM
    if params is None:
        params = {
            'objective': 'binary',
            'metric': 'binary_logloss',
            'boosting_type': 'gbdt',
            'n_jobs': -1,
            'random_state': 42,
            'class_weight': 'balanced'
        }
    clf = lgb.LGBMClassifier(**params)
    
    # Fit
    start_time = time.time()
    clf.fit(X_train_scaled, y_train)
    print(f"LightGBM fitted in {time.time() - start_time:.2f} seconds.")
    
    # Predict
    y_pred = clf.predict(X_test_scaled)
    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    specificity = specificity_score(y_test, y_pred)
    cohen_kappa = cohen_kappa_score(y_test, y_pred)
    
    print("\nEvaluation Metrics:")
    print(f"Accuracy:       {accuracy:.4f}")
    print(f"Precision:      {precision:.4f}")
    print(f"Recall (Wake):  {recall:.4f}")
    print(f"Sensitivity (Sleep): {specificity_score(y_test==0, y_pred==0):.4f}")
    print(f"Specificity (Wake):  {specificity:.4f}")
    print(f"F1 Score:       {f1:.4f}")
    print(f"Cohen Kappa:    {cohen_kappa:.4f}")
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Sleep','Wake'], zero_division=0))
    
    return clf

# -------------------- Hyperparameter tuning with RandomizedSearchCV --------------------
def tune_lgb(X_train, y_train):
    print("\n🔍 Running Random Search for Hyperparameter Tuning (Binary)...")
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    lgb_model = lgb.LGBMClassifier(objective='binary', n_jobs=-1, random_state=42, class_weight='balanced')
    
    param_dist = {
        'num_leaves': [31, 50, 70],
        'max_depth': [-1, 10, 20, 30],
        'learning_rate': [0.01, 0.05, 0.1],
        'n_estimators': [100, 200, 500],
        'min_child_samples': [10, 20, 30]
    }
    
    search = RandomizedSearchCV(
        lgb_model,
        param_distributions=param_dist,
        n_iter=20,
        cv=3,
        scoring='f1_macro',
        verbose=1,
        n_jobs=-1,
        random_state=42
    )
    
    search.fit(X_train_scaled, y_train)
    
    print(f"\n✅ Best hyperparameters: {search.best_params_}")
    print(f"🏅 Best CV score (F1 macro): {search.best_score_:.4f}")
    
    return search.best_params_

# -------------------- Load train/test split --------------------
hdf5_path = r"C:\Users\anita\OneDrive - Universitetet i Oslo\Masteroppgave zzz\UOslo_March2025\Combined\Step4_normalized_train_test_FINAL.h5"

with h5py.File(hdf5_path, 'r') as f:
    X_train = f['X_train'][:]
    X_test = f['X_test'][:]
    y_train = f['y_train'][:]
    y_test = f['y_test'][:]

# -------------------- New 80/20 train/test resplit! --------------------
X_all = np.concatenate([X_train, X_test], axis=0)
y_all = np.concatenate([y_train, y_test], axis=0)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=0.2,
    random_state=42,
    stratify=y_all
)

print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")


# -------------------- Filter unknown labels --------------------
valid_labels = [0,1,2,3,5]
train_idx = np.isin(y_train, valid_labels)
test_idx  = np.isin(y_test, valid_labels)

X_train = X_train[train_idx]
y_train = y_train[train_idx]
X_test  = X_test[test_idx]
y_test  = y_test[test_idx]

# -------------------- Map to binary: Sleep=0, Wake=1 --------------------
y_train_bin = np.copy(y_train)
y_test_bin  = np.copy(y_test)
sleep_stages = [1,2,3,5]
y_train_bin[np.isin(y_train_bin, sleep_stages)] = 0
y_test_bin[np.isin(y_test_bin, sleep_stages)] = 0
y_train_bin[y_train == 0] = 1
y_test_bin[y_test == 0] = 1

# -------------------- Upsample Wake class if needed --------------------
X_train_sleep = X_train[y_train_bin == 0]
y_train_sleep = y_train_bin[y_train_bin == 0]
X_train_wake = X_train[y_train_bin == 1]
y_train_wake = y_train_bin[y_train_bin == 1]

if len(X_train_sleep) > len(X_train_wake):
    X_wake_upsampled, y_wake_upsampled = resample(
        X_train_wake, y_train_wake,
        replace=True,
        n_samples=len(X_train_sleep),
        random_state=42
    )
    X_train_balanced = np.vstack([X_train_sleep, X_wake_upsampled])
    y_train_balanced = np.hstack([y_train_sleep, y_wake_upsampled])
else:
    X_train_balanced = np.vstack([X_train_sleep, X_train_wake])
    y_train_balanced = np.hstack([y_train_sleep, y_train_wake])

print(f"Balanced training set: {X_train_balanced.shape[0]} samples")

# -------------------- Run Random Search --------------------
best_params = tune_lgb(X_train_balanced, y_train_balanced)

# -------------------- Train final LightGBM --------------------
lgb_model = lgb_classifier(
    X_train_balanced, y_train_balanced,
    X_test, y_test_bin,
    use_scaling=True,
    params=best_params
)


Training samples: 46398, Test samples: 11600
Balanced training set: 77662 samples

🔍 Running Random Search for Hyperparameter Tuning (Binary)...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
[LightGBM] [Info] Number of positive: 38831, number of negative: 38831
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.023790 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 32716
[LightGBM] [Info] Number of data points in the train set: 77662, number of used features: 129
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000

✅ Best hyperparameters: {'num_leaves': 70, 'n_estimators': 500, 'min_child_samples': 30, 'max_depth': 20, 'learning_rate': 0.1}
🏅 Best CV score (F1 macro): 0.9832

--- LightGBM for Sleep vs Awake classification ---
Data scaled with StandardScaler.
[LightGBM] [Info] Number of positive: 38831, number of negative: 38831
[LightGBM] [Info] Auto-choosing c

c:\Users\anita\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


##### Test, no wake upsample

In [1]:
import h5py
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report, cohen_kappa_score)
from sklearn.utils import resample
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import time



# -------------------- Specificity calculation --------------------
def specificity_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn = cm[0, 0]
    fp = cm[0, 1]
    return tn / (tn + fp) if (tn + fp) > 0 else 0

# -------------------- LightGBM classifier --------------------
def lgb_classifier(X_train, y_train, X_test, y_test, use_scaling=True, params=None):
    print("\n--- LightGBM for Sleep vs Awake classification ---")
    
    # Scale features if needed
    if use_scaling:
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        print("Data scaled with StandardScaler.")
    else:
        X_train_scaled, X_test_scaled = X_train, X_test
        print("Data scaling skipped.")
    
    # Initialize LightGBM
    if params is None:
        params = {
            'objective': 'binary',
            'metric': 'binary_logloss',
            'boosting_type': 'gbdt',
            'n_jobs': -1,
            'random_state': 42,
            #'class_weight': 'balanced'
        }
    clf = lgb.LGBMClassifier(**params)
    
    # Fit
    start_time = time.time()
    clf.fit(X_train_scaled, y_train)
    print(f"LightGBM fitted in {time.time() - start_time:.2f} seconds.")
    
    # Predict
    y_pred = clf.predict(X_test_scaled)
    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    specificity = specificity_score(y_test, y_pred)
    cohen_kappa = cohen_kappa_score(y_test, y_pred)
    
    print("\nEvaluation Metrics:")
    print(f"Accuracy:       {accuracy:.4f}")
    print(f"Precision:      {precision:.4f}")
    print(f"Recall (Wake):  {recall:.4f}")
    print(f"Sensitivity (Sleep): {specificity_score(y_test==0, y_pred==0):.4f}")
    print(f"Specificity (Wake):  {specificity:.4f}")
    print(f"F1 Score:       {f1:.4f}")
    print(f"Cohen Kappa:    {cohen_kappa:.4f}")
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Sleep','Wake'], zero_division=0))
    
    return clf

# -------------------- Hyperparameter tuning with RandomizedSearchCV --------------------
def tune_lgb(X_train, y_train):
    print("\n🔍 Running Random Search for Hyperparameter Tuning (Binary)...")
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    lgb_model = lgb.LGBMClassifier(objective='binary', n_jobs=-1, random_state=42)
    
    param_dist = {
        'num_leaves': [31, 50, 70],
        'max_depth': [-1, 10, 20, 30],
        'learning_rate': [0.01, 0.05, 0.1],
        'n_estimators': [100, 200, 500],
        'min_child_samples': [10, 20, 30]
    }
    
    search = RandomizedSearchCV(
        lgb_model,
        param_distributions=param_dist,
        n_iter=20,
        cv=3,
        scoring='f1_macro',
        verbose=1,
        n_jobs=-1,
        random_state=42
    )
    
    search.fit(X_train_scaled, y_train)
    
    print(f"\n✅ Best hyperparameters: {search.best_params_}")
    print(f"🏅 Best CV score (F1 macro): {search.best_score_:.4f}")
    
    return search.best_params_

# -------------------- Load train/test split --------------------
hdf5_path = r"C:\Users\anita\OneDrive - Universitetet i Oslo\Masteroppgave zzz\UOslo_March2025\Combined\Step4_normalized_train_test_FINAL.h5"

with h5py.File(hdf5_path, 'r') as f:
    X_train = f['X_train'][:]
    X_test = f['X_test'][:]
    y_train = f['y_train'][:]
    y_test = f['y_test'][:]

# -------------------- New 80/20 train/test resplit! --------------------
X_all = np.concatenate([X_train, X_test], axis=0)
y_all = np.concatenate([y_train, y_test], axis=0)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=0.2,
    random_state=42,
    stratify=y_all
)

print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")


# -------------------- Filter unknown labels --------------------
valid_labels = [0,1,2,3,5]
train_idx = np.isin(y_train, valid_labels)
test_idx  = np.isin(y_test, valid_labels)

X_train = X_train[train_idx]
y_train = y_train[train_idx]
X_test  = X_test[test_idx]
y_test  = y_test[test_idx]

# -------------------- Map to binary: Sleep=0, Wake=1 --------------------
y_train_bin = np.copy(y_train)
y_test_bin  = np.copy(y_test)
sleep_stages = [1,2,3,5]
y_train_bin[np.isin(y_train_bin, sleep_stages)] = 0
y_test_bin[np.isin(y_test_bin, sleep_stages)] = 0
y_train_bin[y_train == 0] = 1
y_test_bin[y_test == 0] = 1


# -------------------- Run Random Search --------------------
best_params = tune_lgb(X_train, y_train_bin)

# -------------------- Train final LightGBM --------------------
lgb_model = lgb_classifier(
    X_train, y_train_bin,
    X_test, y_test_bin,
    use_scaling=True,
    params=best_params
)


Training samples: 46398, Test samples: 11600

🔍 Running Random Search for Hyperparameter Tuning (Binary)...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
[LightGBM] [Info] Number of positive: 7485, number of negative: 38831
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.037896 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 32713
[LightGBM] [Info] Number of data points in the train set: 46316, number of used features: 129
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.161607 -> initscore=-1.646318
[LightGBM] [Info] Start training from score -1.646318

✅ Best hyperparameters: {'num_leaves': 70, 'n_estimators': 500, 'min_child_samples': 20, 'max_depth': 20, 'learning_rate': 0.05}
🏅 Best CV score (F1 macro): 0.9266

--- LightGBM for Sleep vs Awake classification ---
Data scaled with StandardScaler.
[LightGBM] [Info] Number of positive: 7485, number of negative: 38831
[LightGBM] [Info

c:\Users\anita\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


### Hypertuning: Test 1

In [3]:
import h5py
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report, cohen_kappa_score)
from sklearn.utils import resample
from sklearn.model_selection import RandomizedSearchCV
import lightgbm as lgb
import time


# -------------------- Specificity calculation --------------------
def specificity_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn = cm[0, 0]
    fp = cm[0, 1]
    return tn / (tn + fp) if (tn + fp) > 0 else 0

# -------------------- LightGBM classifier --------------------
def lgb_classifier(X_train, y_train, X_test, y_test, use_scaling=True, params=None):
    print("\n--- LightGBM for Sleep vs Awake classification ---")
    
    # Scale features if needed
    if use_scaling:
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        print("Data scaled with StandardScaler.")
    else:
        X_train_scaled, X_test_scaled = X_train, X_test
        print("Data scaling skipped.")
    
    # Initialize LightGBM
    if params is None:
        params = {
            'objective': 'binary',
            'metric': 'binary_logloss',
            'boosting_type': 'gbdt',
            'n_jobs': -1,
            'random_state': 42,
            'class_weight': 'balanced'
        }
    clf = lgb.LGBMClassifier(**params)
    
    # Fit
    start_time = time.time()
    clf.fit(X_train_scaled, y_train)
    print(f"LightGBM fitted in {time.time() - start_time:.2f} seconds.")
    
    # Predict
    y_pred = clf.predict(X_test_scaled)
    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    specificity = specificity_score(y_test, y_pred)
    cohen_kappa = cohen_kappa_score(y_test, y_pred)
    
    print("\nEvaluation Metrics:")
    print(f"Accuracy:       {accuracy:.4f}")
    print(f"Precision:      {precision:.4f}")
    print(f"Recall (Wake):  {recall:.4f}")
    print(f"Sensitivity (Sleep): {specificity_score(y_test==0, y_pred==0):.4f}")
    print(f"Specificity (Wake):  {specificity:.4f}")
    print(f"F1 Score:       {f1:.4f}")
    print(f"Cohen Kappa:    {cohen_kappa:.4f}")
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Sleep','Wake'], zero_division=0))
    
    return clf

# -------------------- Hyperparameter tuning with RandomizedSearchCV --------------------
def tune_lgb(X_train, y_train):
    print("\n🔍 Running Random Search for Hyperparameter Tuning (Binary)...")
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    lgb_model = lgb.LGBMClassifier(objective='binary', n_jobs=-1, random_state=42, class_weight='balanced')
    
    param_dist = {
        'num_leaves': [20, 60, 80],
        'max_depth': [-1, 10, 20, 30],
        'learning_rate': [0.01, 0.05, 0.1],  
        'n_estimators': [100, 200, 500],
        'min_child_samples': [10, 20, 30]
    }
    
    search = RandomizedSearchCV(
        lgb_model,
        param_distributions=param_dist,
        n_iter=20,
        cv=3,
        scoring='f1_macro',
        verbose=1,
        n_jobs=-1,
        random_state=42
    )
    
    search.fit(X_train_scaled, y_train)
    
    print(f"\n✅ Best hyperparameters: {search.best_params_}")
    print(f"🏅 Best CV score (F1 macro): {search.best_score_:.4f}")
    
    return search.best_params_

# -------------------- Load train/test split --------------------
hdf5_path = r"C:\Users\anita\OneDrive - Universitetet i Oslo\Masteroppgave zzz\UOslo_March2025\Combined\Step4_normalized_train_test_FINAL.h5"

with h5py.File(hdf5_path, 'r') as f:
    X_train = f['X_train'][:]
    X_test = f['X_test'][:]
    y_train = f['y_train'][:]
    y_test = f['y_test'][:]

# -------------------- Filter unknown labels --------------------
valid_labels = [0,1,2,3,5]
train_idx = np.isin(y_train, valid_labels)
test_idx  = np.isin(y_test, valid_labels)

X_train = X_train[train_idx]
y_train = y_train[train_idx]
X_test  = X_test[test_idx]
y_test  = y_test[test_idx]

# -------------------- Map to binary: Sleep=0, Wake=1 --------------------
y_train_bin = np.copy(y_train)
y_test_bin  = np.copy(y_test)
sleep_stages = [1,2,3,5]
y_train_bin[np.isin(y_train_bin, sleep_stages)] = 0
y_test_bin[np.isin(y_test_bin, sleep_stages)] = 0
y_train_bin[y_train == 0] = 1
y_test_bin[y_test == 0] = 1

# -------------------- Upsample Wake class if needed --------------------
X_train_sleep = X_train[y_train_bin == 0]
y_train_sleep = y_train_bin[y_train_bin == 0]
X_train_wake = X_train[y_train_bin == 1]
y_train_wake = y_train_bin[y_train_bin == 1]

if len(X_train_sleep) > len(X_train_wake):
    X_wake_upsampled, y_wake_upsampled = resample(
        X_train_wake, y_train_wake,
        replace=True,
        n_samples=len(X_train_sleep),
        random_state=42
    )
    X_train_balanced = np.vstack([X_train_sleep, X_wake_upsampled])
    y_train_balanced = np.hstack([y_train_sleep, y_wake_upsampled])
else:
    X_train_balanced = np.vstack([X_train_sleep, X_train_wake])
    y_train_balanced = np.hstack([y_train_sleep, y_train_wake])

print(f"Balanced training set: {X_train_balanced.shape[0]} samples")

# -------------------- Run Random Search --------------------
best_params = tune_lgb(X_train_balanced, y_train_balanced)

# -------------------- Train final LightGBM --------------------
lgb_model = lgb_classifier(
    X_train_balanced, y_train_balanced,
    X_test, y_test_bin,
    use_scaling=True,
    params=best_params
)

#smaller learning_rate': [0.001, 0.005, 0.01]= little bit lower results
#smaller num_leaves': [10, 30, 50]= pretty much same
#higher num_leaves': [20, 60, 80]= same result


Balanced training set: 77662 samples

🔍 Running Random Search for Hyperparameter Tuning (Binary)...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
[LightGBM] [Info] Number of positive: 38831, number of negative: 38831
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.031470 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 32719
[LightGBM] [Info] Number of data points in the train set: 77662, number of used features: 129
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000

✅ Best hyperparameters: {'num_leaves': 80, 'n_estimators': 500, 'min_child_samples': 30, 'max_depth': 20, 'learning_rate': 0.1}
🏅 Best CV score (F1 macro): 0.9833

--- LightGBM for Sleep vs Awake classification ---
Data scaled with StandardScaler.
[LightGBM] [Info] Number of positive: 38831, number of negative: 38831
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of test

c:\Users\anita\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


# CNN

In [ ]:
import h5py
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim

# -------------------- Device --------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------- Load Data --------------------
hdf5_path = r"C:\Users\anita\OneDrive - Universitetet i Oslo\Masteroppgave zzz\UOslo_March2025\Combined\Step4_normalized_train_test_FINAL.h5"

with h5py.File(hdf5_path, 'r') as f:
    X_train = f['X_train'][:]
    X_test = f['X_test'][:]
    y_train = f['y_train'][:]
    y_test = f['y_test'][:]

# Combine and re-split 80/20
X_all = np.concatenate([X_train, X_test], axis=0)
y_all = np.concatenate([y_train, y_test], axis=0)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=0.2,
    random_state=42,
    stratify=y_all
)

# -------------------- Filter valid labels --------------------
valid_labels = [0,1,2,3,5]
train_idx = np.isin(y_train, valid_labels)
test_idx  = np.isin(y_test, valid_labels)

X_train = X_train[train_idx]
y_train = y_train[train_idx]
X_test  = X_test[test_idx]
y_test  = y_test[test_idx]

# Remap labels to 0–4 for PyTorch
label_map = {0:0, 1:1, 2:2, 3:3, 5:4}

# -------------------- Map to binary: Sleep=0, Wake=1 --------------------#SLEEP VS WAKE
# Original labels: 0=Wake, 1=N1, 2=N2, 3=N3, 5=REM
y_train_bin = np.copy(y_train)
y_test_bin  = np.copy(y_test)

sleep_stages = [1,2,3,5]  # N1, N2, N3, REM
y_train_bin[np.isin(y_train_bin, sleep_stages)] = 0
y_test_bin[np.isin(y_test_bin, sleep_stages)] = 0

y_train_bin[y_train == 0] = 1  # Wake
y_test_bin[y_test == 0] = 1

num_classes = 2  # binary classification

# -------------------- Convert to Torch --------------------
X_train = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
X_test = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)
y_train = torch.tensor(y_train_bin, dtype=torch.long)
y_test = torch.tensor(y_test_bin, dtype=torch.long)

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256)

# -------------------- CNN Model --------------------
model = SleepCNN(input_size=X_train.shape[2], num_classes=num_classes).to(device)

# -------------------- Loss & Optimizer --------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# -------------------- Training --------------------
epochs = 20
start_time = time.time()

for epoch in range(epochs):
    model.train()
    running_loss = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

print(f"\nTraining finished in {time.time()-start_time:.2f} seconds")

# -------------------- Evaluation --------------------
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=['Sleep','Wake'], zero_division=0))

print("Cohen Kappa:", cohen_kappa_score(all_labels, all_preds))

# y_train = np.vectorize(label_map.get)(y_train)
# y_test  = np.vectorize(label_map.get)(y_test)

# num_classes = 5

# # -------------------- Scale --------------------
# scaler = StandardScaler()
# X_train = scaler.fit_transform(X_train)
# X_test = scaler.transform(X_test)

# # -------------------- Convert to Torch --------------------
# X_train = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
# X_test = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)
# y_train = torch.tensor(y_train, dtype=torch.long)
# y_test = torch.tensor(y_test, dtype=torch.long)

# train_dataset = TensorDataset(X_train, y_train)
# test_dataset = TensorDataset(X_test, y_test)

# train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
# test_loader = DataLoader(test_dataset, batch_size=256)

# # -------------------- CNN Model --------------------
# class SleepCNN(nn.Module):
#     def __init__(self, input_size, num_classes):
#         super(SleepCNN, self).__init__()
        
#         self.conv1 = nn.Conv1d(1, 32, kernel_size=5, padding=2)
#         self.bn1 = nn.BatchNorm1d(32)
        
#         self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
#         self.bn2 = nn.BatchNorm1d(64)
        
#         self.pool = nn.MaxPool1d(2)
#         self.relu = nn.ReLU()
#         self.dropout = nn.Dropout(0.5)
        
#         # Dynamically compute flattened size
#         with torch.no_grad():
#             dummy = torch.zeros(1, 1, input_size)
#             dummy = self.pool(self.relu(self.bn1(self.conv1(dummy))))
#             dummy = self.pool(self.relu(self.bn2(self.conv2(dummy))))
#             self.flattened_size = dummy.numel()
        
#         self.fc1 = nn.Linear(self.flattened_size, 128)
#         self.fc2 = nn.Linear(128, num_classes)

#     def forward(self, x):
#         x = self.pool(self.relu(self.bn1(self.conv1(x))))
#         x = self.pool(self.relu(self.bn2(self.conv2(x))))
#         x = torch.flatten(x, 1)
#         x = self.dropout(self.relu(self.fc1(x)))
#         x = self.fc2(x)
#         return x

# input_size = X_train.shape[2]
# model = SleepCNN(input_size, num_classes).to(device)

# # -------------------- Loss & Optimizer --------------------
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.parameters(), lr=0.001)

# # -------------------- Training --------------------
# epochs = 20
# start_time = time.time()

# for epoch in range(epochs):
#     model.train()
#     running_loss = 0
    
#     for inputs, labels in train_loader:
#         inputs, labels = inputs.to(device), labels.to(device)
        
#         optimizer.zero_grad()
#         outputs = model(inputs)
#         loss = criterion(outputs, labels)
#         loss.backward()
#         optimizer.step()
        
#         running_loss += loss.item()
    
#     print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

# print(f"\nTraining finished in {time.time()-start_time:.2f} seconds")

# # -------------------- Evaluation --------------------
# model.eval()
# all_preds = []
# all_labels = []

# with torch.no_grad():
#     for inputs, labels in test_loader:
#         inputs = inputs.to(device)
#         outputs = model(inputs)
#         preds = torch.argmax(outputs, dim=1).cpu().numpy()
        
#         all_preds.extend(preds)
#         all_labels.extend(labels.numpy())

# # Map labels back to original for reporting
# reverse_map = {0:0, 1:1, 2:2, 3:3, 4:5}
# all_preds = np.vectorize(reverse_map.get)(all_preds)
# all_labels = np.vectorize(reverse_map.get)(all_labels)

# print("\nClassification Report:")
# print(classification_report(all_labels, all_preds, zero_division=0))

# print("Cohen Kappa:", cohen_kappa_score(all_labels, all_preds))

Using device: cpu
Epoch [1/20], Loss: 0.2441
Epoch [2/20], Loss: 0.1822
Epoch [3/20], Loss: 0.1645
Epoch [4/20], Loss: 0.1550
Epoch [5/20], Loss: 0.1460
Epoch [6/20], Loss: 0.1446
Epoch [7/20], Loss: 0.1370
Epoch [8/20], Loss: 0.1313
Epoch [9/20], Loss: 0.1299
Epoch [10/20], Loss: 0.1238
Epoch [11/20], Loss: 0.1198
Epoch [12/20], Loss: 0.1173
Epoch [13/20], Loss: 0.1146
Epoch [14/20], Loss: 0.1104
Epoch [15/20], Loss: 0.1100
Epoch [16/20], Loss: 0.1065
Epoch [17/20], Loss: 0.1032
Epoch [18/20], Loss: 0.1019
Epoch [19/20], Loss: 0.0995
Epoch [20/20], Loss: 0.0956

Training finished in 101.30 seconds

Classification Report:
              precision    recall  f1-score   support

       Sleep       0.98      0.96      0.97      9708
        Wake       0.82      0.88      0.85      1872

    accuracy                           0.95     11580
   macro avg       0.90      0.92      0.91     11580
weighted avg       0.95      0.95      0.95     11580

Cohen Kappa: 0.819403910391052


### CNN + LSTM

In [7]:
import h5py
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import classification_report, cohen_kappa_score
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
import time

# -------------------- Device --------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# -------------------- Load HDF5 --------------------
hdf5_path = r"C:\Users\anita\OneDrive - Universitetet i Oslo\Masteroppgave zzz\UOslo_March2025\Combined\Step4_normalized_train_test_FINAL.h5"
with h5py.File(hdf5_path, 'r') as f:
    X_train = f['X_train'][:]
    X_test  = f['X_test'][:]
    y_train = f['y_train'][:]
    y_test  = f['y_test'][:]

# -------------------- Filter valid labels --------------------
valid_labels = [0,1,2,3,5]
train_idx = np.isin(y_train, valid_labels)
test_idx  = np.isin(y_test, valid_labels)
X_train = X_train[train_idx]
y_train = y_train[train_idx]
X_test  = X_test[test_idx]
y_test  = y_test[test_idx]

# -------------------- Map to binary: Sleep=0, Wake=1 --------------------
sleep_stages = [1,2,3,5]
y_train_bin = np.copy(y_train)
y_test_bin  = np.copy(y_test)
y_train_bin[np.isin(y_train_bin, sleep_stages)] = 0
y_test_bin[np.isin(y_test_bin, sleep_stages)] = 0
y_train_bin[y_train == 0] = 1
y_test_bin[y_test == 0] = 1

# -------------------- Standardize features --------------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# -------------------- Sequence building (for LSTM) --------------------
SEQ_LEN = 20  # temporal window (number of consecutive epochs)
def build_sequences(X, y, seq_len=SEQ_LEN):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_len + 1):
        X_seq.append(X[i:i+seq_len])
        y_seq.append(y[i+seq_len-1])
    return np.array(X_seq), np.array(y_seq)

X_train_seq, y_train_seq = build_sequences(X_train, y_train_bin)
X_test_seq, y_test_seq   = build_sequences(X_test, y_test_bin)

# -------------------- Convert to Torch --------------------
X_train_seq = torch.tensor(X_train_seq, dtype=torch.float32).to(DEVICE)
X_test_seq  = torch.tensor(X_test_seq, dtype=torch.float32).to(DEVICE)
y_train_seq = torch.tensor(y_train_seq, dtype=torch.long).to(DEVICE)
y_test_seq  = torch.tensor(y_test_seq, dtype=torch.long).to(DEVICE)

train_ds = TensorDataset(X_train_seq, y_train_seq)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)

# -------------------- CNN-LSTM Model --------------------
class CNN_LSTM(nn.Module):
    def __init__(self, n_features, seq_len, hidden_size=128, num_classes=2):
        super().__init__()
        # CNN part
        self.conv1 = nn.Conv1d(1, 32, kernel_size=5, padding=2)
        self.bn1   = nn.BatchNorm1d(32)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.bn2   = nn.BatchNorm1d(64)
        self.pool  = nn.MaxPool1d(2)
        self.relu  = nn.ReLU()
        
        # Dynamically compute flattened size for LSTM input
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_features)
            dummy = self.pool(self.relu(self.bn1(self.conv1(dummy))))
            dummy = self.pool(self.relu(self.bn2(self.conv2(dummy))))
            self.cnn_out_size = dummy.shape[1]  # number of features after CNN
        
        # LSTM part
        self.lstm = nn.LSTM(input_size=self.cnn_out_size,
                            hidden_size=hidden_size,
                            num_layers=1,
                            batch_first=True,
                            bidirectional=True)
        self.fc = nn.Linear(hidden_size*2, num_classes)
    
    def forward(self, x):
    # x: [batch, seq_len, n_features]
        batch, seq_len, n_feat = x.size()
        x = x.view(batch*seq_len, 1, n_feat)           # merge batch & seq_len for CNN
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))  # [batch*seq_len, channels, seq_len_after_pool]

        # Take only the mean across the time dimension (seq_len_after_pool) or flatten per channel
        x = x.mean(dim=2)  # now x.shape = [batch*seq_len, channels]  ✅ channels = 64

        # Reshape for LSTM: [batch, seq_len, input_size]
        x = x.view(batch, seq_len, -1)  # -1 = channels = 64, matches LSTM input_size

        out, _ = self.lstm(x)
        out = out[:, -1, :]  # take last timestep
        return self.fc(out)

model = CNN_LSTM(n_features=X_train_seq.shape[2], seq_len=SEQ_LEN, hidden_size=128, num_classes=2).to(DEVICE)

# -------------------- Loss & Optimizer --------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# -------------------- Training --------------------
EPOCHS = 15
start_time = time.time()

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.4f}")

print(f"\nTraining finished in {time.time()-start_time:.2f} seconds")

# -------------------- Evaluation --------------------
model.eval()
with torch.no_grad():
    logits = model(X_test_seq)
    preds = torch.argmax(logits, dim=1).cpu().numpy()
    y_true = y_test_seq.cpu().numpy()

print("\n--- Classification Report (Wake vs Sleep) ---")
print(classification_report(y_true, preds, target_names=["Sleep","Wake"]))
print("Cohen Kappa:", cohen_kappa_score(y_true, preds))

Using device: cpu
Epoch 1/15, Loss: 0.4193
Epoch 2/15, Loss: 0.2950
Epoch 3/15, Loss: 0.2517
Epoch 4/15, Loss: 0.2262
Epoch 5/15, Loss: 0.2155
Epoch 6/15, Loss: 0.2092
Epoch 7/15, Loss: 0.2050
Epoch 8/15, Loss: 0.2007
Epoch 9/15, Loss: 0.1972
Epoch 10/15, Loss: 0.1946
Epoch 11/15, Loss: 0.1915
Epoch 12/15, Loss: 0.1889
Epoch 13/15, Loss: 0.1858
Epoch 14/15, Loss: 0.1836
Epoch 15/15, Loss: 0.1821

Training finished in 2410.06 seconds

--- Classification Report (Wake vs Sleep) ---
              precision    recall  f1-score   support

       Sleep       0.94      0.97      0.95      9690
        Wake       0.81      0.69      0.74      1871

    accuracy                           0.92     11561
   macro avg       0.88      0.83      0.85     11561
weighted avg       0.92      0.92      0.92     11561

Cohen Kappa: 0.698396475394665
